In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.options.display.max_columns=1000

**Merge files**

In [ ]:
n_batches = 130

df = pd.concat((
    pd.read_parquet(f"02_features/batch_{i}.parquet")
    for i in range(1, n_batches + 1)
))

In [ ]:
df = df[ df["MaxMoveNumber"] >= 10 ]

In [ ]:
n_train = 1_000_000
n_test = 250_000

#shuffle
df = df.sample(n_train+n_test, random_state=1660)

df_train = df.head(n_train)
df_test = df.tail(n_test)

In [ ]:
len(df_train), len(df_test)

In [ ]:
df_train.head()

**Check one feature**

In [ ]:
feature = (df_train["MaxMoveNumber"] // 2).clip(0, 40)

In [ ]:
fig = px.line(
    feature.value_counts().sort_index()
)

fig.data[0].mode = "lines+markers"
fig.update_layout(
    template="plotly_white",
    showlegend=False
)

fig.show()

In [ ]:
fig = px.line(
    df_train.groupby(feature).agg({"Elo": "mean"})
)

fig.data[0].mode = "lines+markers"
fig.update_layout(
    template="plotly_white",
    showlegend=False
)

fig.show()

**Saving files**

In [ ]:
def bin_features(df):
    
    df_new = pd.DataFrame()

    # remove outliers
    # TODO anomaly detection (?)
    for feature in df.columns:
        if feature in ["GameId", "White", "Black", "WhiteElo", "BlackElo", "Elo", "Opening", "ECO", "FirstMoves"]:
            df_new[feature] = df[feature]
        else:
            df_new[feature] = df[feature].clip(
                df[feature].quantile(0.005), 
                df[feature].quantile(0.995)
            )
    
    return df_new

In [ ]:
bin_features(df_train).to_parquet("03_datasets/binned_train.parquet")
bin_features(df_test).to_parquet("03_datasets/binned_test.parquet")